# 🍎 Fruits & Vegetables AI – Block 4: NLP (Recipe & Health Advisor)

## Objective
Use GPT-4o-mini to generate personalized recipe suggestions and health advice
based on the CV classification result and ML-predicted nutritional values.

## Integration
- **Input from CV block:** detected food label (e.g. 'apple')
- **Input from ML block:** predicted calories per 100g
- **Output:** personalized recipe suggestion + health advice in English

## NLP Approach
We use **prompt engineering** with GPT-4o-mini (zero-shot).
Three prompt iterations are compared using a structured evaluation framework
across four criteria: Completeness, Structure, Actionability, and Hallucination Risk.

In [21]:
#import os
#from openai import OpenAI
#from dotenv import load_dotenv

#load_dotenv()
#client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
#print('✅ OpenAI client ready')

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)   # ← override=True erzwingt das Neuladen

api_key = os.environ.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# Kontrolle: zeigt die letzten 6 Zeichen des Keys
print(f'✅ Key geladen, endet auf: ...{api_key[-6:]}')

✅ Key geladen, endet auf: ...gVB6EA


## 1. Three Prompt Iterations

We design three prompts of increasing sophistication:

| Iteration | Strategy | Key characteristics |
| --- | --- | --- |
| **Iteration 1** | Minimal prompt | No context, no structure, one sentence |
| **Iteration 2** | Structured prompt | Role assignment, nutritional context, structured output |
| **Iteration 3** | Optimised prompt | Role + context + strict format + grounding instruction |

In [23]:
# ── Iteration 1: Minimal prompt ───────────────────────────────────────────────
def prompt_v1(food_label: str, calories: float) -> str:
    """Iteration 1: One-sentence prompt, no context, no structure."""
    prompt = f'Give me a recipe for {food_label} which has {calories} calories per 100g.'
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=250,
        temperature=0.7
    )
    return response.choices[0].message.content


# ── Iteration 2: Structured prompt ───────────────────────────────────────────
def prompt_v2(food_label: str, calories: float,
              protein: float, fat: float, carbs: float, fiber: float) -> str:
    """Iteration 2: Role assignment + nutritional context + structured output."""
    prompt = f"""You are a professional nutritionist and chef.

A user uploaded an image and the AI system detected: **{food_label}**

Nutritional values per 100g (predicted by ML model):
- Calories: {calories} kcal
- Protein: {protein}g | Fat: {fat}g | Carbs: {carbs}g | Fiber: {fiber}g

Please provide:
1. **Health Benefits** (2-3 sentences)
2. **Best Recipe** (1 simple recipe, max 5 steps)
3. **Serving Tip** (1 short tip)

Keep it friendly and concise."""

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=400,
        temperature=0.7
    )
    return response.choices[0].message.content


# ── Iteration 3: Optimised prompt ────────────────────────────────────────────
def prompt_v3(food_label: str, calories: float,
              protein: float, fat: float, carbs: float, fiber: float) -> str:
    """Iteration 3: Role + context + strict format + grounding instruction.
    Key improvements over v2:
    - System message separates role from task
    - Explicit grounding: base advice on the provided numbers, not general knowledge
    - Strict output format to reduce hallucination
    - Temperature lowered to 0.3 for more consistent, fact-grounded output
    """
    system_msg = (
        'You are a certified nutritionist and chef. '
        'Always base your nutritional advice strictly on the values provided. '
        'Do not invent or estimate nutritional data. '
        'Be concise, factual, and practical.'
    )
    user_msg = f"""The AI system detected: {food_label}

Measured nutritional values per 100g:
- Calories: {calories} kcal
- Protein: {protein}g | Fat: {fat}g | Carbohydrates: {carbs}g | Fiber: {fiber}g

Respond EXACTLY in this format:

HEALTH BENEFITS:
[2 evidence-based sentences referencing the values above]

RECIPE: [Name]
Ingredients: [max 5 items]
Steps:
1. [step]
2. [step]
3. [step]

SERVING TIP:
[1 practical tip]"""

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': user_msg}
        ],
        max_tokens=400,
        temperature=0.3
    )
    return response.choices[0].message.content


print('✅ Three prompt functions defined')

✅ Three prompt functions defined


## 2. Run All Three Prompts on the Same Input

We test all three iterations on identical inputs so the comparison is fair.

In [24]:
# ── Test case 1: Apple ────────────────────────────────────────────────────────
TEST_FOOD     = 'apple'
TEST_CALORIES = 52.0
TEST_PROTEIN  = 0.3
TEST_FAT      = 0.2
TEST_CARBS    = 14.0
TEST_FIBER    = 2.4

print('=' * 60)
print(f'Input: {TEST_FOOD} | {TEST_CALORIES} kcal | '
      f'P:{TEST_PROTEIN}g F:{TEST_FAT}g C:{TEST_CARBS}g Fi:{TEST_FIBER}g')
print('=' * 60)

print('\n>>> ITERATION 1: Minimal Prompt')
print('-' * 40)
out_v1_apple = prompt_v1(TEST_FOOD, TEST_CALORIES)
print(out_v1_apple)

print('\n>>> ITERATION 2: Structured Prompt')
print('-' * 40)
out_v2_apple = prompt_v2(TEST_FOOD, TEST_CALORIES,
                         TEST_PROTEIN, TEST_FAT, TEST_CARBS, TEST_FIBER)
print(out_v2_apple)

print('\n>>> ITERATION 3: Optimised Prompt')
print('-' * 40)
out_v3_apple = prompt_v3(TEST_FOOD, TEST_CALORIES,
                         TEST_PROTEIN, TEST_FAT, TEST_CARBS, TEST_FIBER)
print(out_v3_apple)

Input: apple | 52.0 kcal | P:0.3g F:0.2g C:14.0g Fi:2.4g

>>> ITERATION 1: Minimal Prompt
----------------------------------------
Certainly! Here's a simple recipe that features apples and keeps the calorie count around 52 calories per 100g. This recipe is for a **Fresh Apple Salad**. 

### Fresh Apple Salad

#### Ingredients:
- 200g fresh apples (any variety you prefer, such as Granny Smith, Fuji, or Gala)
- 50g celery, diced
- 30g walnuts, chopped (optional)
- 1 tablespoon lemon juice
- 1 teaspoon honey (optional)
- A pinch of cinnamon (optional)

#### Instructions:

1. **Prepare the Apples**: 
   - Wash the apples thoroughly. Core and chop them into bite-sized pieces. Leave the skin on for added nutrients and texture.

2. **Combine Ingredients**:
   - In a mixing bowl, combine the chopped apples and diced celery. If you're using walnuts, add them to the mix.

3. **Dress the Salad**: 
   - In a small bowl, mix the lemon juice and honey (if using). Drizzle this dressing over the appl

In [25]:
# ── Test case 2: Broccoli (more nutritionally complex) ────────────────────────
print('=' * 60)
print('Input: broccoli | 34 kcal | P:2.8g F:0.4g C:7.0g Fi:2.6g')
print('=' * 60)

print('\n>>> ITERATION 1: Minimal Prompt')
print('-' * 40)
out_v1_broc = prompt_v1('broccoli', 34)
print(out_v1_broc)

print('\n>>> ITERATION 2: Structured Prompt')
print('-' * 40)
out_v2_broc = prompt_v2('broccoli', 34, 2.8, 0.4, 7.0, 2.6)
print(out_v2_broc)

print('\n>>> ITERATION 3: Optimised Prompt')
print('-' * 40)
out_v3_broc = prompt_v3('broccoli', 34, 2.8, 0.4, 7.0, 2.6)
print(out_v3_broc)

Input: broccoli | 34 kcal | P:2.8g F:0.4g C:7.0g Fi:2.6g

>>> ITERATION 1: Minimal Prompt
----------------------------------------
Certainly! Broccoli is a nutritious vegetable that is low in calories, making it a great addition to any meal. Here's a simple recipe for steamed broccoli that keeps it healthy and delicious while maintaining a low calorie count.

### Steamed Broccoli Recipe

#### Ingredients:
- 100g fresh broccoli florets
- 1 tsp olive oil (optional)
- Salt and pepper to taste
- Lemon juice (optional, for serving)

#### Instructions:

1. **Prepare the Broccoli:**
   - Wash the broccoli florets under cold water to remove any dirt or impurities.
   - Cut the broccoli into bite-sized pieces if needed.

2. **Steam the Broccoli:**
   - Fill a pot with about an inch of water and bring it to a boil.
   - Place the broccoli in a steamer basket and set it over the boiling water. Cover the pot with a lid.
   - Steam the broccoli for about 4-5 minutes, or until it is tender but still

## 3. Structured Evaluation

We evaluate all three iterations across four criteria on a 1–3 scale:

| Criterion | 1 (Poor) | 2 (Good) | 3 (Excellent) |
| --- | --- | --- | --- |
| **Completeness** | Missing key sections | Most sections present | All sections present |
| **Structure** | Free-form, hard to parse | Partially structured | Clear, consistent format |
| **Actionability** | Vague advice | Somewhat specific | Concrete, immediately usable |
| **Hallucination Risk** | Invents nutrition data | Mixes provided/generic data | Strictly grounded in input |

### Evaluation Results

| Criterion | Iteration 1 | Iteration 2 | Iteration 3 |
| --- | --- | --- | --- |
| Completeness | 1 | 3 | 3 |
| Structure | 1 | 2 | 3 |
| Actionability | 1 | 2 | 3 |
| Hallucination Risk | 1 | 2 | 3 |
| **Total** | **4/12** | **9/12** | **12/12** |

### Analysis per Iteration

**Iteration 1 — Minimal Prompt (score: 4/12)**  
The one-sentence prompt produces a generic recipe but ignores all nutritional context.
The model invents nutritional claims from general knowledge rather than using the
ML-predicted values. Output format is unpredictable and hard to display in the app.
No health advice, no serving tip. The calorie value is acknowledged but not used meaningfully.

**Iteration 2 — Structured Prompt (score: 9/12)**  
Adding a role ('professional nutritionist'), the full macronutrient context, and a
numbered output structure produces dramatically better results. All three sections
(Health Benefits, Recipe, Serving Tip) are consistently present. However, the model
sometimes supplements the provided values with generic knowledge (e.g. citing vitamins
not present in the input), which introduces mild hallucination risk.

**Iteration 3 — Optimised Prompt (score: 12/12)**  
Separating role into a system message, adding an explicit grounding instruction
('base advice strictly on the values provided, do not invent data'), enforcing a
strict output template, and reducing temperature to 0.3 produces the most consistent
and reliable output. The format is machine-parseable and directly renderable in the
Gradio interface. Hallucination risk is minimised because the model is explicitly
instructed not to supplement with outside knowledge.

### Selected Prompt for Deployment

**Iteration 3** is used in the deployed Gradio app (`app.py`) because it produces
the most structured, grounded, and consistent output. The lower temperature (0.3 vs 0.7)
also improves reproducibility, which is important for a production application.

In [26]:
# ── Quantitative comparison: response length and consistency ──────────────────
import re

def count_sections(text: str) -> int:
    """Count how many expected sections are present in the output."""
    keywords = ['health', 'recipe', 'serving', 'benefit', 'tip', 'ingredient', 'step']
    return sum(1 for kw in keywords if kw.lower() in text.lower())

comparison = {
    'Iteration': ['V1 (Minimal)', 'V2 (Structured)', 'V3 (Optimised)'],
    'Word count (apple)': [
        len(out_v1_apple.split()),
        len(out_v2_apple.split()),
        len(out_v3_apple.split())
    ],
    'Word count (broccoli)': [
        len(out_v1_broc.split()),
        len(out_v2_broc.split()),
        len(out_v3_broc.split())
    ],
    'Sections found (apple)': [
        count_sections(out_v1_apple),
        count_sections(out_v2_apple),
        count_sections(out_v3_apple)
    ],
    'Has system role': ['No', 'No', 'Yes'],
    'Grounding instruction': ['No', 'No', 'Yes'],
    'Temperature': [0.7, 0.7, 0.3]
}

import pandas as pd
comp_df = pd.DataFrame(comparison)
print('=== Prompt Comparison Summary ===')
print(comp_df.to_string(index=False))

=== Prompt Comparison Summary ===
      Iteration  Word count (apple)  Word count (broccoli)  Sections found (apple) Has system role Grounding instruction  Temperature
   V1 (Minimal)                 174                    188                       2              No                    No          0.7
V2 (Structured)                 114                    143                       5              No                    No          0.7
 V3 (Optimised)                 116                    120                       7             Yes                   Yes          0.3


## 4. Integration with CV and ML Blocks

The NLP block receives its inputs directly from the other two blocks:

```
CV Block (CLIP)          ML Block (Ridge)         NLP Block (GPT-4o-mini)
──────────────           ──────────────────       ───────────────────────
Image → food_label  →    food_label + lookup  →   food_label +
         + confidence    → calories (predicted)    calories +
                         + macronutrients          macronutrients
                                                   → health advice +
                                                     recipe + serving tip
```

The NLP component serves two key roles in the system:

1. **Interpretation:** Translates raw numbers (52 kcal, 14g carbs) into
   human-understandable health insights that a non-expert user can act on.

2. **Error mitigation:** If the CV block misclassifies (e.g. pepper → tomato),
   the recipe advice will be for tomato — still useful and safe, just not optimal.
   The confidence score from CV (visible in the app) helps the user judge reliability.

## 5. Limitations

- **API dependency:** The NLP block requires an active OpenAI API key and internet
  connection. If the API is unavailable, the app cannot generate advice.
- **Language:** All three prompts generate English output. A multilingual extension
  (e.g. detecting user locale) would improve accessibility.
- **Hallucination risk (residual):** Even Iteration 3 may occasionally cite benefits
  not directly derivable from the input features (e.g. vitamin C content for broccoli
  is not in our feature set). This is a known LLM limitation.
- **No retrieval (RAG):** The current implementation uses pure prompt engineering.
  A RAG extension with a nutrition knowledge base would improve factual accuracy.